# Pele: reprodução do treino com EfficientNet-B0

**Este notebook faz um treino novo e real. Os resultados são desta reprodução, não do modelo `pele_cor` que está no app.**
O código original do ensemble foi perdido. Aqui repetimos o que o pacote registra: EfficientNet-B0, 224x224, normalização ImageNet,
5 folds por `lesion_id`, calibração de Platt, limiar escolhido fora do teste, benigno x maligno (`mel` + `bcc`) e `akiec` fora.
O número de épocas, a taxa de aprendizado e a função de perda do original não foram registrados: os valores abaixo são escolhas nossas.

**Como rodar**
1. Coloque o dataset em `data/Skin_Cancer` (as subpastas podem ter qualquer nome).
2. Rode tudo com `teste_rapido = True`. Leva poucos minutos e confere GPU, pastas e gráficos.
3. Mude para `teste_rapido = False` e rode tudo de novo. A primeira época mostra a previsão de tempo.

Se o PC desligar, rode tudo de novo: os folds já concluídos são aproveitados. Só o fold interrompido recomeça.

In [ ]:
import csv
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

exigir_gpu = True
if exigir_gpu and not torch.cuda.is_available():
    raise RuntimeError(f"GPU não encontrada. Confira o kernel .venv e o PyTorch instalado ({torch.__version__}).")
device = "cuda" if torch.cuda.is_available() else "cpu"
nome_gpu = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print("Dispositivo:", nome_gpu)

In [ ]:
# Primeiro rode com True (poucas imagens, 2 folds e 1 época). O resultado fica em outra pasta e não entra no resultados.csv
teste_rapido = True

DATA_DIR = Path("../data/Skin_Cancer")

epocas = 12
paciencia = 4  # para o fold se a AUC de validação não melhorar nesse número de épocas
batch = 32  # se der "CUDA out of memory", use 16
lr = 3e-4
weight_decay = 1e-4
n_folds = 5
seed = 42
tamanho = 224
imagens_por_diagnostico = None

if teste_rapido:
    epocas, n_folds, imagens_por_diagnostico = 1, 2, 60

positivos = ["bcc", "mel"]
negativos = ["bkl", "df", "nv", "vasc"]  # akiec fica de fora, como no pacote original
class_names = ["benigno", "maligno"]
media = [0.485, 0.456, 0.406]
desvio = [0.229, 0.224, 0.225]

ROTULO = "REPRODUÇÃO DO TREINO, não é o modelo do app"
nome = "Pele_EfficientNet_reproducao" + ("_TESTE_RAPIDO" if teste_rapido else "")
output_dir = Path("../resultados/Pele") / nome
modelos_dir = Path("../modelos_pt") / nome  # pasta própria: não sobrescreve nenhum outro modelo
# O teste rápido não pode se misturar à tabela compartilhada pelos notebooks
results_csv = output_dir / "resultados_teste_rapido.csv" if teste_rapido else Path("../graficos/resultados.csv")
dataset = "Pele cor EfficientNet-B0 binario reproducao"

config = dict(epocas=epocas, paciencia=paciencia, batch=batch, lr=lr, weight_decay=weight_decay, n_folds=n_folds, seed=seed,
              tamanho=tamanho, imagens_por_diagnostico=imagens_por_diagnostico)
output_dir.mkdir(parents=True, exist_ok=True)
modelos_dir.mkdir(parents=True, exist_ok=True)

# Retomar com outra configuração misturaria folds diferentes
arquivo_config = output_dir / "config.json"
if arquivo_config.exists() and json.loads(arquivo_config.read_text(encoding="utf-8")) != config:
    raise RuntimeError(f"A configuração mudou desde a última execução. Para recomeçar, apague {output_dir} e {modelos_dir}.")
arquivo_config.write_text(json.dumps(config, indent=2), encoding="utf-8")

In [ ]:
def carregar_dados(data_dir):
    """Lê o metadata e guarda as imagens na memória, já em 224x224: cerca de 1,5 GB de RAM e bem menos leitura de disco."""
    arquivos_meta = sorted(data_dir.rglob("HAM10000_metadata*.csv"))
    if not arquivos_meta:
        raise FileNotFoundError(f"HAM10000_metadata.csv não encontrado em {data_dir.resolve()}")
    meta = pd.read_csv(arquivos_meta[0])

    # O Kaggle traz as pastas repetidas em maiúsculas e minúsculas: cada imagem é usada uma vez só
    caminhos = {}
    for caminho in sorted(data_dir.rglob("*.jpg")):
        caminhos.setdefault(caminho.stem.lower(), caminho)

    meta = meta[meta["dx"].isin(positivos + negativos)].copy()
    meta["caminho"] = meta["image_id"].str.lower().map(caminhos)
    faltando = int(meta["caminho"].isna().sum())
    if faltando:
        raise FileNotFoundError(f"{faltando} imagens do metadata não foram encontradas em {data_dir.resolve()}")

    meta = meta.sort_values("image_id").reset_index(drop=True)
    if imagens_por_diagnostico is not None:
        meta = meta.groupby("dx", group_keys=False).head(imagens_por_diagnostico).reset_index(drop=True)

    imagens = np.empty((len(meta), tamanho, tamanho, 3), dtype=np.uint8)
    for i, caminho in enumerate(meta["caminho"]):
        with Image.open(caminho) as imagem:
            imagens[i] = np.asarray(imagem.convert("RGB").resize((tamanho, tamanho), Image.Resampling.BICUBIC))
        if (i + 1) % 2000 == 0:
            print(f"{i + 1}/{len(meta)} imagens lidas")
    alvos = meta["dx"].isin(positivos).astype(int).to_numpy()
    return imagens, alvos, meta["dx"].to_numpy(), meta["lesion_id"].to_numpy(), meta["image_id"].to_numpy()


inicio_carga = time.perf_counter()
imagens, alvos, diagnosticos, lesoes, ids = carregar_dados(DATA_DIR)
print(f"Imagens: {len(alvos)} | Lesões: {len(np.unique(lesoes))} | Malignas: {alvos.mean():.1%} | Leitura: {time.perf_counter() - inicio_carga:.0f} s")
print(pd.Series(diagnosticos).value_counts().to_dict())

In [ ]:
# Separação por lesão: todas as fotos de uma lesão ficam no mesmo lado. O teste não participa do treino, da calibração nem do limiar
divisor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
treino_idx, teste_idx = next(divisor.split(imagens, diagnosticos, lesoes))

folds = list(StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed).split(treino_idx, diagnosticos[treino_idx], lesoes[treino_idx]))
folds = [(treino_idx[tr], treino_idx[va]) for tr, va in folds]

auditoria = {
    "imagens_treino": len(treino_idx),
    "imagens_teste": len(teste_idx),
    "lesoes_treino": int(len(np.unique(lesoes[treino_idx]))),
    "lesoes_teste": int(len(np.unique(lesoes[teste_idx]))),
    "lesoes_em_comum": int(len(set(lesoes[treino_idx]) & set(lesoes[teste_idx]))),
    "lesoes_em_comum_por_fold": [int(len(set(lesoes[tr]) & set(lesoes[va]))) for tr, va in folds],
    "maligno_treino_%": round(float(alvos[treino_idx].mean() * 100), 2),
    "maligno_teste_%": round(float(alvos[teste_idx].mean() * 100), 2),
}
assert auditoria["lesoes_em_comum"] == 0 and not any(auditoria["lesoes_em_comum_por_fold"])
(output_dir / "auditoria_divisao.json").write_text(json.dumps(auditoria, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame({"image_id": ids[teste_idx], "dx": diagnosticos[teste_idx], "lesion_id": lesoes[teste_idx]}).to_csv(output_dir / "imagens_teste.csv", index=False)
print(auditoria)

In [ ]:
transformar_treino = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.06, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(media, desvio),
])
transformar_avaliacao = transforms.Compose([transforms.ToTensor(), transforms.Normalize(media, desvio)])


class ImagensPele(Dataset):
    def __init__(self, indices, transformar):
        self.indices = indices
        self.transformar = transformar

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        indice = self.indices[i]
        return self.transformar(Image.fromarray(imagens[indice])), float(alvos[indice])


def criar_modelo():
    modelo = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)  # precisa de internet na primeira vez (~20 MB)
    modelo.classifier[1] = nn.Linear(modelo.classifier[1].in_features, 1)
    return modelo.to(device)


def calcular_logits(modelo, indices):
    modelo.eval()
    # num_workers=0: no Windows, os workers não enxergam classes definidas dentro do notebook
    carregador = DataLoader(ImagensPele(indices, transformar_avaliacao), batch_size=batch * 2, shuffle=False, num_workers=0)
    logits = []
    with torch.inference_mode(), torch.amp.autocast("cuda", enabled=device == "cuda"):
        for x, _ in carregador:
            logits.append(modelo(x.to(device)).reshape(-1).float().cpu().numpy())
    return np.concatenate(logits)


def perda_bce(logits, y, peso_positivo):
    criterio = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(peso_positivo))
    return float(criterio(torch.from_numpy(logits), torch.from_numpy(y.astype(np.float32))))


def treinar_fold(numero, tr_idx, va_idx):
    torch.manual_seed(seed + numero)
    np.random.seed(seed + numero)
    modelo = criar_modelo()

    y_treino = alvos[tr_idx]
    peso_positivo = float((y_treino == 0).sum() / max((y_treino == 1).sum(), 1))  # compensa o desbalanceamento
    criterio = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(peso_positivo, device=device))
    otimizador = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=weight_decay)
    agendador = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=epocas)
    escalador = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    carregador = DataLoader(ImagensPele(tr_idx, transformar_treino), batch_size=batch, shuffle=True, num_workers=0, drop_last=len(tr_idx) > batch)

    historico, melhor_auc, melhor_estado, melhor_epoca = [], -1.0, None, 0
    inicio = time.perf_counter()
    for epoca in range(1, epocas + 1):
        modelo.train()
        soma_perda, acertos, vistos = 0.0, 0, 0
        for x, y in carregador:
            x, y = x.to(device), y.float().to(device)
            otimizador.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=device == "cuda"):
                saida = modelo(x).reshape(-1)
                perda = criterio(saida, y)
            escalador.scale(perda).backward()
            escalador.step(otimizador)
            escalador.update()
            soma_perda += perda.item() * len(y)
            acertos += ((saida > 0).float() == y).sum().item()
            vistos += len(y)
        agendador.step()

        logits_val = calcular_logits(modelo, va_idx)
        y_val = alvos[va_idx]
        auc = float(roc_auc_score(y_val, logits_val))
        historico.append({
            "fold": numero,
            "epoca": epoca,
            "perda_treino": soma_perda / vistos,
            "acc_treino": acertos / vistos,
            "perda_validacao": perda_bce(logits_val, y_val, peso_positivo),
            "auc_validacao": auc,
            "acc_balanceada_validacao": float(balanced_accuracy_score(y_val, logits_val > 0)),
        })
        if auc > melhor_auc:
            melhor_auc, melhor_epoca = auc, epoca
            melhor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}

        decorrido = time.perf_counter() - inicio
        h = historico[-1]
        print(f"Fold {numero} | Época {epoca}/{epocas} | perda {h['perda_treino']:.4f} | val {h['perda_validacao']:.4f} | "
              f"AUC val {auc:.4f} | {decorrido / epoca:.0f} s/época | fold termina em até {decorrido / epoca * (epocas - epoca) / 60:.0f} min")
        if epoca - melhor_epoca >= paciencia:
            print(f"Parada antecipada: sem melhora desde a época {melhor_epoca}")
            break

    modelo.load_state_dict(melhor_estado)
    return modelo, historico, melhor_epoca, time.perf_counter() - inicio

In [ ]:
# Cada fold é salvo ao terminar. Ao rodar de novo, os folds prontos são lidos do disco
resultados_folds = []
for numero, (tr_idx, va_idx) in enumerate(folds, start=1):
    arquivo_fold = output_dir / f"fold_{numero}.json"
    arquivo_logits = output_dir / f"fold_{numero}_logits.npz"
    if arquivo_fold.exists() and arquivo_logits.exists():
        print(f"Fold {numero}: já concluído, aproveitando")
    else:
        print(f"===== Fold {numero}/{n_folds} | treino {len(tr_idx)} | validação {len(va_idx)} =====")
        modelo, historico, melhor_epoca, tempo = treinar_fold(numero, tr_idx, va_idx)
        np.savez(arquivo_logits, val=calcular_logits(modelo, va_idx), teste=calcular_logits(modelo, teste_idx))
        torch.save({"state_dict": modelo.cpu().state_dict(), "architecture": "efficientnet_b0", "fold": numero, "config": config}, modelos_dir / f"fold_{numero}_best.pt")
        # Grava por último e troca de nome de uma vez: se o PC desligar antes, o fold conta como não concluído
        temporario = arquivo_fold.with_suffix(".tmp")
        temporario.write_text(json.dumps({"historico": historico, "melhor_epoca": melhor_epoca, "tempo_s": tempo}), encoding="utf-8")
        temporario.replace(arquivo_fold)
        del modelo
        if device == "cuda":
            torch.cuda.empty_cache()
    dados = json.loads(arquivo_fold.read_text(encoding="utf-8"))
    logits = np.load(arquivo_logits)
    resultados_folds.append({**dados, "logits_val": logits["val"], "logits_teste": logits["teste"]})

historico = pd.DataFrame([linha for fold in resultados_folds for linha in fold["historico"]])
historico.to_csv(output_dir / "historico_epocas.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
tempo_total = sum(fold["tempo_s"] for fold in resultados_folds)
print(f"Tempo de treino: {time.strftime('%H:%M:%S', time.gmtime(tempo_total))}")

In [ ]:
# Calibração de Platt com a validação de cada fold; o limiar sai da validação cruzada, sem olhar o teste
sigmoide = lambda v: 1.0 / (1.0 + np.exp(-np.clip(v, -50, 50)))

prob_val_por_fold, prob_teste_por_fold, calibracoes = [], [], []
for fold, (_, va_idx) in zip(resultados_folds, folds):
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(fold["logits_val"].reshape(-1, 1), alvos[va_idx])
    inclinacao, intercepto = float(platt.coef_[0][0]), float(platt.intercept_[0])
    calibracoes.append({"inclinacao": inclinacao, "intercepto": intercepto})
    prob_val_por_fold.append(sigmoide(inclinacao * fold["logits_val"] + intercepto))
    prob_teste_por_fold.append(sigmoide(inclinacao * fold["logits_teste"] + intercepto))

y_oof = np.concatenate([alvos[va] for _, va in folds])
prob_oof = np.concatenate(prob_val_por_fold)
limiares = np.round(np.arange(0.01, 1.0, 0.01), 2)


def desempenho_limiar(limiar):
    prev = prob_oof >= limiar
    sensibilidade = (prev & (y_oof == 1)).sum() / max((y_oof == 1).sum(), 1)
    return balanced_accuracy_score(y_oof, prev), sensibilidade


limiar = float(max(limiares, key=desempenho_limiar))  # empate: fica o de maior sensibilidade

acc_scores = [accuracy_score(alvos[va], p >= limiar) for (_, va), p in zip(folds, prob_val_por_fold)]
f1_scores = [f1_score(alvos[va], p >= limiar, average="macro") for (_, va), p in zip(folds, prob_val_por_fold)]
auc_folds = [roc_auc_score(alvos[va], p) for (_, va), p in zip(folds, prob_val_por_fold)]

# Teste: média das probabilidades calibradas dos modelos dos folds, como no ensemble original
y_teste = alvos[teste_idx]
prob_teste = np.mean(prob_teste_por_fold, axis=0)
prev_teste = (prob_teste >= limiar).astype(int)
vn, fp, fn, vp = confusion_matrix(y_teste, prev_teste, labels=[0, 1]).ravel()

metricas_teste = {
    "descricao": ROTULO,
    "imagens_teste": len(y_teste),
    "limiar": limiar,
    "acuracia_%": accuracy_score(y_teste, prev_teste) * 100,
    "acuracia_chutando_sempre_benigno_%": (y_teste == 0).mean() * 100,
    "acuracia_balanceada_%": balanced_accuracy_score(y_teste, prev_teste) * 100,
    "sensibilidade_%": vp / max(vp + fn, 1) * 100,
    "especificidade_%": vn / max(vn + fp, 1) * 100,
    "f1_macro_%": f1_score(y_teste, prev_teste, average="macro") * 100,
    "auc_roc": roc_auc_score(y_teste, prob_teste),
    "auc_pr": average_precision_score(y_teste, prob_teste),
    "brier": brier_score_loss(y_teste, prob_teste),
    "verdadeiros_positivos": int(vp),
    "falsos_negativos": int(fn),
    "verdadeiros_negativos": int(vn),
    "falsos_positivos": int(fp),
    "auc_validacao_media": float(np.mean(auc_folds)),
    "auc_validacao_desvio": float(np.std(auc_folds)),
}
pd.Series(metricas_teste).to_frame("valor").to_csv(output_dir / "metricas_teste.csv", sep=";", decimal=",", encoding="utf-8-sig")

por_diagnostico = pd.DataFrame({"dx": diagnosticos[teste_idx], "real": y_teste, "previsto": prev_teste})
por_diagnostico = por_diagnostico.assign(acerto=por_diagnostico["real"] == por_diagnostico["previsto"]).groupby("dx").agg(imagens=("acerto", "size"), acerto_pct=("acerto", lambda a: a.mean() * 100))
por_diagnostico.to_csv(output_dir / "acerto_por_diagnostico.csv", sep=";", decimal=",", encoding="utf-8-sig")
(output_dir / "calibracao_e_limiar.json").write_text(json.dumps({"limiar": limiar, "folds": calibracoes}, indent=2), encoding="utf-8")

for chave, valor in metricas_teste.items():
    print(f"{chave}: {valor:.3f}" if isinstance(valor, float) else f"{chave}: {valor}")
print(por_diagnostico.round(1))

In [ ]:
def salvar(fig, nome_arquivo):
    fig.suptitle(fig._suptitle.get_text() if fig._suptitle else ROTULO, fontsize=11)
    fig.tight_layout()
    fig.savefig(output_dir / nome_arquivo, dpi=200, bbox_inches="tight")
    plt.show()


# Curvas por época: estes gráficos só existem porque houve treino real
fig, (ax_perda, ax_auc) = plt.subplots(1, 2, figsize=(13, 4.8))
for numero, grupo in historico.groupby("fold"):
    cor = plt.cm.tab10(numero - 1)
    ax_perda.plot(grupo["epoca"], grupo["perda_treino"], color=cor, label=f"Fold {numero} treino", marker="o", ms=3)
    ax_perda.plot(grupo["epoca"], grupo["perda_validacao"], color=cor, linestyle="--", label=f"Fold {numero} validação", marker="o", ms=3)
    ax_auc.plot(grupo["epoca"], grupo["auc_validacao"], color=cor, label=f"Fold {numero}", marker="o", ms=3)
    melhor = resultados_folds[numero - 1]["melhor_epoca"]
    ax_auc.scatter([melhor], grupo.loc[grupo["epoca"] == melhor, "auc_validacao"], color=cor, s=70, edgecolor="black", zorder=3)
ax_perda.set(xlabel="Época", ylabel="Perda (BCE)", title="Perda por época")
ax_auc.set(xlabel="Época", ylabel="AUC", title="AUC de validação (círculo maior: época escolhida)")
for eixo in (ax_perda, ax_auc):
    eixo.grid(alpha=0.3)
    eixo.legend(fontsize=7, ncols=2)
fig.suptitle(f"{ROTULO} | {nome_gpu}")
salvar(fig, "curvas_treino.png")

# Matriz de confusão do teste
fig, eixos = plt.subplots(1, 2, figsize=(10, 4.2))
matriz = confusion_matrix(y_teste, prev_teste, labels=[0, 1])
ConfusionMatrixDisplay(matriz, display_labels=class_names).plot(ax=eixos[0], cmap="Blues", colorbar=False, values_format="d")
eixos[0].set(title=f"Teste: contagem (limiar {limiar:.2f})", xlabel="Previsto", ylabel="Real")
ConfusionMatrixDisplay(matriz / np.maximum(matriz.sum(axis=1, keepdims=True), 1) * 100, display_labels=class_names).plot(ax=eixos[1], cmap="Blues", colorbar=False, values_format=".1f")
eixos[1].set(title="Teste: % de cada classe real", xlabel="Previsto", ylabel="Real")
fig.suptitle(ROTULO)
salvar(fig, "matriz_confusao_teste.png")

# Curva ROC e ponto de operação
fig, eixo = plt.subplots(figsize=(5.6, 5))
RocCurveDisplay.from_predictions(y_teste, prob_teste, ax=eixo, name="Ensemble (teste)")
eixo.scatter([1 - metricas_teste["especificidade_%"] / 100], [metricas_teste["sensibilidade_%"] / 100], color="red", zorder=3, label=f"Limiar {limiar:.2f}")
eixo.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
eixo.set(xlabel="1 - especificidade", ylabel="Sensibilidade", title="Curva ROC")
eixo.legend(loc="lower right")
eixo.grid(alpha=0.3)
fig.suptitle(ROTULO)
salvar(fig, "roc_teste.png")

# Calibração: mostra se 30% previsto corresponde a cerca de 30% de casos malignos
fig, eixo = plt.subplots(figsize=(5.6, 5))
fracao, previsto = calibration_curve(y_teste, prob_teste, n_bins=10, strategy="quantile")
eixo.plot(previsto, fracao, marker="o", label="Ensemble (teste)")
eixo.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="Calibração perfeita")
eixo.set(xlabel="Probabilidade prevista", ylabel="Fração real de malignos", title="Calibração")
eixo.legend()
eixo.grid(alpha=0.3)
fig.suptitle(ROTULO)
salvar(fig, "calibracao_teste.png")

# Acerto por diagnóstico: separa os erros em mel e bcc, e as classes pequenas
fig, eixo = plt.subplots(figsize=(7, 4))
barras = eixo.bar(por_diagnostico.index, por_diagnostico["acerto_pct"], color=["#c0392b" if dx in positivos else "#2980b9" for dx in por_diagnostico.index])
eixo.bar_label(barras, labels=[f"{p:.1f}% (n={n})" for p, n in zip(por_diagnostico["acerto_pct"], por_diagnostico["imagens"])], fontsize=8)
eixo.set(ylim=(0, 112), ylabel="Acerto no teste (%)", title="Acerto por diagnóstico (vermelho: maligno; azul: benigno)")
fig.suptitle(ROTULO)
salvar(fig, "acerto_por_diagnostico.png")
print("Gráficos em:", output_dir.resolve())

In [ ]:
# Mesmas colunas dos outros notebooks. acc e f1 vêm da validação por fold; as métricas do teste estão em metricas_teste.csv
def num(valor):
    return f"{valor:.2f}".replace(".", ",")


contagem = pd.Series(np.array(class_names)[alvos]).value_counts()
linha = {
    "data_hora": time.strftime("%Y-%m-%d %H:%M:%S"),
    "dataset": dataset,
    "modo": "RGB",
    "normalizado": "Sim",
    "imagens_total": len(alvos),
    "imagens_por_classe": " | ".join(f"{c}: {int(contagem.get(c, 0))}" for c in class_names),
    "tamanho": f"{tamanho}x{tamanho}",
    "epocas": epocas,
    "batch": batch,
    "lr": str(lr).replace(".", ","),
    "folds": f"{len(folds)}/{n_folds}",
    "gpu": nome_gpu,
    "tempo_total": time.strftime("%H:%M:%S", time.gmtime(tempo_total)),
    "acc_media_%": num(np.mean(acc_scores) * 100),
    "acc_desvio_%": num(np.std(acc_scores) * 100),
    "f1_media_%": num(np.mean(f1_scores) * 100),
    "f1_desvio_%": num(np.std(f1_scores) * 100),
    "acc_por_fold_%": " | ".join(num(a * 100) for a in acc_scores),
    "f1_por_fold_%": " | ".join(num(f * 100) for f in f1_scores),
    "pasta_graficos": str(output_dir),
}
results_csv.parent.mkdir(parents=True, exist_ok=True)
novo = not results_csv.exists()
with open(results_csv, "a", newline="", encoding="utf-8-sig") as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=list(linha), delimiter=";")
    if novo:
        escritor.writeheader()
    escritor.writerow(linha)
print("Resultado adicionado em:", results_csv.resolve())